# WhyBook Phase 2 Fine-Tuning (Kaggle)

This notebook fine-tunes WhyBook on the Phase 1 NCERT chemistry dataset using Unsloth + QLoRA.

Expected input: one uploaded `.jsonl` file in `/kaggle/input/...` with exact keys:
`concept`, `chapter`, `class`, `what`, `why`, `real_world`.

Output artifacts:
- LoRA adapters
- GGUF quantized model (`Q4_K_M`)
- Optional Hugging Face upload

In [1]:
import os
# Let it use BOTH T4s (needed for float32 Gemma 4)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # REMOVED
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_USE_FUSED_CE_LOSS"] = "0"


In [2]:
%%capture
!pip install --upgrade pip -q
!pip install --no-deps bitsandbytes accelerate peft trl unsloth_zoo -q
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer -q
!pip install --no-deps unsloth -q
!pip install llama-cpp-python -q
!pip install --upgrade transformers -q

In [3]:
# transformers is already installed in setup cell above

Note: you may need to restart the kernel to use updated packages.


In [4]:
import glob
import json
import gc
import shutil
from pathlib import Path

import torch

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")

# HuggingFace token
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded ✓")
except Exception:
    print("⚠ No HF_TOKEN found in Kaggle Secrets")

# Directories
WORKDIR = Path("/kaggle/working")
DATA_DIR = WORKDIR / "data"
LORA_DIR = WORKDIR / "models" / "lora"
MERGED_DIR = WORKDIR / "models" / "merged"
GGUF_DIR = WORKDIR / "models" / "gguf"
CHECKPOINT_DIR = WORKDIR / "whybook-e2b-checkpoints"

for p in [DATA_DIR, LORA_DIR, MERGED_DIR, GGUF_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Directories ready ✓")


GPU: Tesla T4 | VRAM: 15.6 GB
HF_TOKEN loaded ✓
Directories ready ✓


In [5]:
MODEL_ID = "google/gemma-4-E2B-it"
MAX_SEQ_LEN = 512
LOAD_IN_4BIT = True
RANDOM_STATE = 42

R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 5
WEIGHT_DECAY = 0.01

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

jsonl_candidates = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
assert jsonl_candidates, "No .jsonl found under /kaggle/input. Upload dataset first."

preferred = [p for p in jsonl_candidates
             if p.endswith("data.jsonl") or p.endswith("ncert_chemistry_whybook.jsonl")]
DATASET_PATH = preferred[0] if preferred else jsonl_candidates[0]
print(f"Using dataset: {DATASET_PATH}")


Using dataset: /kaggle/input/datasets/chandansatwani/ncert-9th-10th-chemistry/data.jsonl


In [6]:
def load_jsonl(path: str) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def validate_record(record: dict) -> tuple[bool, str]:
    required = ["concept", "chapter", "class", "what", "why", "real_world"]
    for field in required:
        if field not in record or not str(record[field]).strip():
            return False, f"Missing or empty field: {field}"

    if len(record["what"].split()) > 100:
        return False, "what field exceeds 100 words"
    if len(record["why"].split()) > 100:
        return False, "why field exceeds 100 words"
    if len(record["real_world"].split()) > 150:
        return False, "real_world field exceeds 150 words"

    refusal_phrases = [
        "i cannot", "i don't know", "as an ai", "i'm not able",
        "it is used in industry", "it has many applications",
        "in conclusion", "to summarize",
    ]
    for field in ["what", "why", "real_world"]:
        text = record[field].lower()
        for phrase in refusal_phrases:
            if phrase in text:
                return False, f"Refusal phrase in {field}: '{phrase}'"

    abstract_phrases = [
        "various industries", "many applications", "widely used",
        "plays an important role", "has many uses",
    ]
    for phrase in abstract_phrases:
        if phrase in record["real_world"].lower():
            return False, f"Abstract phrase in real_world: '{phrase}'"

    if record["what"].strip() == record["why"].strip():
        return False, "what and why are identical"
    if record["why"].strip() == record["real_world"].strip():
        return False, "why and real_world are identical"

    if len(record["what"].split()) < 20:
        return False, "what too short"
    if len(record["why"].split()) < 20:
        return False, "why too short"
    if len(record["real_world"].split()) < 30:
        return False, "real_world too short"

    return True, "ok"

raw_data = load_jsonl(DATASET_PATH)
valid_data = []
invalid = 0

for row in raw_data:
    ok, reason = validate_record(row)
    if ok:
        valid_data.append(row)
    else:
        invalid += 1

assert valid_data, "No valid records after validation"
raw_data = valid_data
print(f"Valid: {len(raw_data)} | Rejected: {invalid}")

Total: 90 | Invalid: 0


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=RANDOM_STATE,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.3f}%)")
print(f"VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `model.base_model.model.model.language_model` require gradients
Trainable: 31,039,488 / 4,204,600,864 (0.738%)
VRAM: 12.14 GB


In [8]:
from datasets import Dataset

def build_text(record: dict) -> dict:
    messages = [
        {"role": "user", "content": [{"type": "text", "text": (
            "You are WhyBook, an NCERT Chemistry tutor for Indian students.\n"
            f"A student wants to understand: {record['concept']} "
            f"from {record['chapter']} (Class {record['class']}).\n"
            "Explain clearly - what it is, why it is taught, "
            "and where the student will see it in real life."
        )}]},
        {"role": "assistant", "content": [{"type": "text", "text": (
            f"What it is:\n{record['what']}\n\n"
            f"Why it is in your textbook:\n{record['why']}\n\n"
            f"Where you will see it in real life:\n{record['real_world']}"
        )}]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

text_rows = [build_text(r) for r in raw_data]
split = int(0.9 * len(text_rows))
train_ds = Dataset.from_list(text_rows[:split])
eval_ds = Dataset.from_list(text_rows[split:])

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print("\n--- Sample (first 500 chars) ---")
print(train_ds[0]["text"][:500])


Train: 81 | Eval: 9

--- Sample (first 500 chars) ---
<bos><|turn>user
You are WhyBook, an NCERT Chemistry tutor for Indian students.
A student wants to understand: Matter from MATTER IN OUR SURROUNDINGS (Class 9).
Explain clearly - what it is, why it is taught, and where the student will see it in real life.<turn|>
<|turn>model
What it is:
Matter is anything that takes up space and has weight. Scientists use this name for all the material that makes up the universe around us. It can be something as large as a star or as tiny as a single particle o


In [10]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_8bit",
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    seed=RANDOM_STATE,
    report_to="none",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_num_proc=1,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    formatting_func=lambda ex: ex["text"],
)

trainer_stats = trainer.train()
print(f"\n✓ Done. Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

eval_losses = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
print(f"Eval losses: {eval_losses}")
print(f"Best: {min(eval_losses):.4f}")


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/81 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/9 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Epoch,Training Loss,Validation Loss
1,4.513007,3.556859
2,2.893896,2.764627
3,2.425377,2.595030



✓ Done. Peak VRAM: 12.14 GB
Eval losses: [3.556859254837036, 2.7646267414093018, 2.595029830932617]
Best: 2.5950


In [12]:
sft_config.num_train_epochs = 6

trainer_stats = trainer.train(resume_from_checkpoint=True)

eval_losses = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
print("Eval losses:", eval_losses)
print(f"Best: {min(eval_losses):.4f}")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81 | Num Epochs = 6 | Total steps = 66
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Epoch,Training Loss,Validation Loss
4,2.277431,2.381398
5,1.964720,2.258987
6,1.769489,2.233464


Eval losses: [3.556859254837036, 2.7646267414093018, 2.595029830932617, 2.3813984394073486, 2.2589874267578125, 2.233464002609253]
Best: 2.2335


In [15]:
sft_config.num_train_epochs = 9  # total 9 (continues from 6)
trainer_stats = trainer.train(resume_from_checkpoint=True)

eval_losses = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
print("Eval losses:", eval_losses)
print(f"Best: {min(eval_losses):.4f}")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81 | Num Epochs = 9 | Total steps = 99
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Epoch,Training Loss,Validation Loss
7,1.697302,2.134725
8,1.508122,2.192601
9,1.414751,2.172212


Eval losses: [3.556859254837036, 2.7646267414093018, 2.595029830932617, 2.3813984394073486, 2.2589874267578125, 2.233464002609253, 2.1347246170043945, 2.192601203918457, 2.1722121238708496]
Best: 2.1347


In [13]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": [{"type": "text", "text": 
    "You are WhyBook, an NCERT Chemistry tutor for Indian students.\n"
    "A student wants to understand: NaCl (Sodium Chloride) from Acids, Bases and Salts (Class 10).\n"
    "Explain clearly - what it is, why it is taught, and where the student will see it in real life."
}]}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs, max_new_tokens=300,
    do_sample=False, pad_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))


Hello! I'm WhyBook, and I'd be happy to help you understand $\text{NaCl}$ (Sodium Chloride) from the Acids, Bases, and Salts chapter.

Here is a clear explanation to help you grasp the concept:

---

## $\text{NaCl}$ (Sodium Chloride) Explained

### 1. What is $\text{NaCl}$?

* **Definition:** $\text{NaCl}$ is a common ionic compound formed from the reaction between a strong acid (like $\text{HCl}$) and a strong base (like $\text{NaOH}$).
* **Properties:**
    * **Nature:** It is a **neutral salt**. When dissolved in water, it dissociates into $\text{Na}^+$ (sodium ion) and $\text{Cl}^-$ (chloride ion), both of which are spectator ions (they don't react further with water).
    * **Solubility:** It is highly soluble in water, meaning it dissolves completely to form a homogeneous solution.
    * **Taste:** It has a **salty taste**.

### 2. Why is $\text{NaCl}$ Taught in Chemistry?

$\text{NaCl}$ is a fundamental compound taught in this chapter for several reasons:

* **Understanding Ion

In [14]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": [{"type": "text", "text": 
    "You are WhyBook, an NCERT Chemistry tutor for Indian students.\n"
    "A student wants to understand: NaCl (Sodium Chloride) from Acids, Bases and Salts (Class 10).\n"
    "Explain clearly - what it is, why it is taught, and where the student will see it in real life."
}]}]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

# Test 1: WITH LoRA adapters
model.enable_adapter_layers()
out_lora = model.generate(input_ids=inputs, max_new_tokens=50, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
text_lora = tokenizer.decode(out_lora[0][inputs.shape[-1]:], skip_special_tokens=True)

# Test 2: WITHOUT LoRA adapters (base model only)
model.disable_adapter_layers()
out_base = model.generate(input_ids=inputs, max_new_tokens=50, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
text_base = tokenizer.decode(out_base[0][inputs.shape[-1]:], skip_special_tokens=True)

# Re-enable
model.enable_adapter_layers()

print("=== WITH LoRA (first 50 tokens) ===")
print(text_lora)
print("\n=== WITHOUT LoRA (first 50 tokens) ===")
print(text_base)
print(f"\n=== Are they identical? {text_lora == text_base} ===")


=== WITH LoRA (first 50 tokens) ===
Hello! I'm WhyBook, and I'd be happy to help you understand $\text{NaCl}$ (Sodium Chloride) from the Acids, Bases, and Salts chapter.

Here is a clear explanation to help you grasp the concept:

=== WITHOUT LoRA (first 50 tokens) ===
Hello! I'm WhyBook, and I'm happy to help you understand **NaCl (Sodium Chloride)** from the perspective of your Class 10 Chemistry syllabus. This is a very important topic, and I will explain it clearly so you

=== Are they identical? False ===


In [16]:
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print(f"✓ LoRA saved to {LORA_DIR}")


✓ LoRA saved to /kaggle/working/models/lora


In [17]:
# Free disk space
for p in [str(CHECKPOINT_DIR), "/root/.cache/huggingface/hub",
          str(MERGED_DIR), str(GGUF_DIR)]:
    if Path(p).exists():
        shutil.rmtree(p, ignore_errors=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)
gc.collect()
torch.cuda.empty_cache()

_, _, free = shutil.disk_usage("/kaggle/working")
print(f"Disk free: {free/1e9:.1f} GB")

# Export GGUF to /tmp (more disk space)
TMP_GGUF = "/tmp/whybook-gguf"
Path(TMP_GGUF).mkdir(parents=True, exist_ok=True)

model.save_pretrained_gguf(
    TMP_GGUF, tokenizer,
    quantization_method="q4_k_m",
    temporary_location="/tmp",
)

# Copy final GGUF to /kaggle/working
for f in list(Path(TMP_GGUF).rglob("*.gguf")) + list(Path(TMP_GGUF + "_gguf").rglob("*.gguf")):
    if "Q4_K_M" in f.name:
        dest = GGUF_DIR / f.name
        shutil.copy2(str(f), str(dest))
        print(f"✓ {f.name}: {f.stat().st_size/1e9:.2f} GB → {dest}")

print(f"\nGGUF files: {list(GGUF_DIR.glob('*.gguf'))}")


Disk free: 20.8 GB
Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:34<00:00, 94.87s/it]


Unsloth: Merge process complete. Saved to `/tmp/whybook-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/whybook-gguf_gguf/gemma-4-E2B-it.F16.gguf', '/tmp/whybook-gguf_gguf/gemma-4-E2B-it.F16-mmproj.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/whybook-gguf_gguf/gemma-4-E2B-it.Q4_K_M.gguf', '/tmp/whybook-gguf_gguf/gemma-4-E2B-it.F16-mmproj.gguf']
Unsloth: No Ollama template mapping found for model 'google/gemma-4-E2B-it'. Skipping Ollama Modelfile


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m /tmp/whybook-gguf_gguf/gemma-4-E2B-it.Q4_K_M.gguf --mmproj /tmp/whybook-gguf_gguf/gemma-4-E2B-it.F16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
✓ gemma-4-E2B-it.Q4_K_M.gguf: 3.43 GB → /kaggle/working/mod

In [18]:
from huggingface_hub import HfApi, login

login(token=os.environ["HF_TOKEN"])
api = HfApi()

HF_USERNAME = "Stinger2311"  # <-- CHANGE THIS
REPO_NAME = f"{HF_USERNAME}/whybook-gemma4-e2b-gguf"

api.create_repo(repo_id=REPO_NAME, exist_ok=True)

for gguf_file in GGUF_DIR.glob("*.gguf"):
    print(f"Uploading {gguf_file.name}...")
    api.upload_file(
        path_or_fileobj=str(gguf_file),
        path_in_repo=gguf_file.name,
        repo_id=REPO_NAME,
    )

api.upload_folder(folder_path=str(LORA_DIR), path_in_repo="lora", repo_id=REPO_NAME)
print(f"\n✓ Done! https://huggingface.co/{REPO_NAME}")


Uploading gemma-4-E2B-it.Q4_K_M.gguf...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✓ Done! https://huggingface.co/Stinger2311/whybook-gemma4-e2b-gguf
